# 🧠 CanScan — Détection de Tumeurs Cérébrales avec YOLOv8 (Kaggle)

Ce notebook implémente l'ensemble de la chaîne de traitement et d'entraînement pour la détection d'objets avec **YOLOv8** sur le jeu de données d'IRM cérébrales de **CanScan**.

---

### 🎯 Objectifs et Exigences Réalisés :
1. **Décompression & Localisation** automatique du dataset zippé (`canscan_detection.zip`).
2. **Vérification d'appariement** : pour chaque image, vérifier l'existence d'un fichier `.txt` correspondant dans les annotations (par nom de base).
3. **Organisation des données** dans les répertoires `images/` et `labels/` avec respect des ensembles `train`, `val` et `test`.
4. **Copie conjointe** de chaque image et de son annotation vers le split approprié.
5. **Élimination / Quarantine** des images orphelines (sans label) et des labels orphelins (sans image).
6. **Validation du format YOLO** : vérification de la structure à 5 valeurs (`class_id x y w h`), de la cohérence des classes et de la normalisation $[0, 1]$.
7. **Comptage & Statistiques** : nombre d'images, de labels et répartition des classes par split.
8. **Visualisation d'échantillons** avec affichage des boîtes englobantes (*bounding boxes*) et noms de classes réelles.
9. **Génération de `data.yaml`** avec chemins absolus, nombre de classes et noms ordonnés.
10. **Configuration des augmentations de données** adaptées aux images médicales (IRM).
11. **Entraînement YOLOv8** avec détection automatique du matériel (GPU/VRAM) et choix des hyperparamètres adaptés.
12. **Évaluation complète** : Précision, Rappel, mAP@0.5, mAP@0.5:0.95 globale et par classe.
13. **Analyse de généralisation** sur l'ensemble de test (matrices de confusion, comparaison Train/Val/Test, détection de surapprentissage).
14. **Sauvegarde des artefacts** : poids `best.pt`, export ONNX, `model_info.json`, prédictions visuelles et archive ZIP téléchargeable.

## 1. Environnement, Dépendances & Vérification Matérielle

In [ ]:
# =============================================================================
# 1. Installation & Imports
# =============================================================================
!pip install -q ultralytics pyyaml pandas matplotlib seaborn opencv-python onnx

import os
import sys
import shutil
import random
import zipfile
from pathlib import Path
from collections import Counter
from datetime import datetime
import json

import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

import ultralytics
from ultralytics import YOLO

# Fixer la graine aléatoire pour la reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Ultralytics Version : {ultralytics.__version__}")
print(f"PyTorch Version     : {torch.__version__}")

# Détection matérielle détaillée
if torch.cuda.is_available():
    gpu_props = torch.cuda.get_device_properties(0)
    vram_gb = gpu_props.total_memory / (1024 ** 3)
    print(f"\n✅ GPU Détecté : {gpu_props.name} | VRAM : {vram_gb:.2f} GB | Cores CPU : {os.cpu_count()}")
else:
    print("\n⚠️ Aucun GPU détecté. En mode Kaggle, activez le GPU (Settings -> Accelerator -> GPU T4 x2 ou P100).")

## 2. Localisation & Extraction du Dataset Zippé (`canscan_detection.zip`)
Cette cellule recherche automatiquement le fichier ZIP ou le dossier existant dans `/kaggle/input/`, `/kaggle/working/` ou en local, puis le décompresse dans le dossier de travail.

In [ ]:
# =============================================================================
# 2. Découverte et Décompression du Dataset
# =============================================================================
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("artifacts/kaggle_run").resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

EXTRACT_DIR = WORK_DIR / "extracted_raw"
DATASET_YOLO_DIR = WORK_DIR / "dataset_yolo"
ARTIFACTS_DIR = WORK_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

def locate_zip_or_dataset():
    """Recherche canscan_detection.zip ou le dossier images/labels dans les chemins courants."""
    search_roots = ["/kaggle/input", ".", "data", ".."]
    # 1. Chercher d'abord un fichier zip
    for root in search_roots:
        r_path = Path(root)
        if not r_path.exists():
            continue
        for p in r_path.rglob("*.zip"):
            if "canscan" in p.name.lower() or "detection" in p.name.lower():
                return "zip", p
    # 2. Chercher un dossier déjà extrait contenant 'images' et 'labels'
    for root in search_roots:
        r_path = Path(root)
        if not r_path.exists():
            continue
        for p in r_path.rglob("detection"):
            if (p / "images").is_dir() and (p / "labels").is_dir():
                return "dir", p
    return None, None

kind, found_path = locate_zip_or_dataset()
print(f"Type trouvé : {kind} -> {found_path}")

if kind == "zip":
    print(f"📦 Extraction de {found_path} vers {EXTRACT_DIR}...")
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(found_path, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    SOURCE_RAW_DIR = EXTRACT_DIR
    print("✅ Extraction terminée avec succès.")
elif kind == "dir":
    SOURCE_RAW_DIR = found_path
    print(f"✅ Utilisation du répertoire existant : {SOURCE_RAW_DIR}")
else:
    raise FileNotFoundError("Impossible de trouver 'canscan_detection.zip' ou le dossier de détection.")

# Localiser la racine effective contenant images/ et labels/
def find_yolo_root(start_dir):
    for root, dirs, _ in os.walk(start_dir):
        if "images" in dirs and "labels" in dirs:
            return Path(root)
    return Path(start_dir)

DETECTED_ROOT = find_yolo_root(SOURCE_RAW_DIR)
print(f"📁 Racine YOLO détectée : {DETECTED_ROOT}")

## 3. Vérification des Paires Images-Annotations & Écartement des Orphelins
Exigences couvertes :
- Vérifier l'existence d'un fichier `.txt` pour chaque image en comparant le nom de base (`stem`).
- Écarter les images sans annotation et les annotations sans image correspondante.
- Organiser proprement les données dans `dataset_yolo/images/{train,val,test}` et `dataset_yolo/labels/{train,val,test}`.

In [ ]:
# =============================================================================
# 3. Vérification, Nettoyage et Organisation des Données
# =============================================================================
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IGNORED_LABEL_FILES = {"classes.txt"}
SPLITS = ["train", "val", "test"]

# Création de l'arborescence cible standard YOLO
for s in SPLITS:
    (DATASET_YOLO_DIR / "images" / s).mkdir(parents=True, exist_ok=True)
    (DATASET_YOLO_DIR / "labels" / s).mkdir(parents=True, exist_ok=True)

def list_split_files(img_dir, lbl_dir):
    """Retourne la liste des images et labels d'un dossier donné."""
    images = [p for p in sorted(img_dir.iterdir()) if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]
    labels = [p for p in sorted(lbl_dir.iterdir()) if p.is_file() and p.suffix.lower() == ".txt" and p.name not in IGNORED_LABEL_FILES]
    return images, labels

audit_records = []
unpaired_log = []

for split in SPLITS:
    # Vérifier si les dossiers existent dans la source détectée
    src_img = DETECTED_ROOT / "images" / split
    src_lbl = DETECTED_ROOT / "labels" / split
    
    # Fallback pour val/valid
    if not src_img.is_dir() and split == "val":
        src_img = DETECTED_ROOT / "images" / "valid"
        src_lbl = DETECTED_ROOT / "labels" / "valid"
        
    if not src_img.is_dir() or not src_lbl.is_dir():
        print(f"⚠️ Split [{split}] absent de la source ({src_img})")
        continue
        
    images, labels = list_split_files(src_img, src_lbl)
    img_stems = {p.stem: p for p in images}
    lbl_stems = {p.stem: p for p in labels}
    
    common_stems = sorted(set(img_stems.keys()) & set(lbl_stems.keys()))
    orphan_images = [img_stems[s] for s in sorted(set(img_stems.keys()) - set(lbl_stems.keys()))]
    orphan_labels = [lbl_stems[s] for s in sorted(set(lbl_stems.keys()) - set(img_stems.keys()))]
    
    # Copie des paires valides vers le répertoire cible
    dest_img_dir = DATASET_YOLO_DIR / "images" / split
    dest_lbl_dir = DATASET_YOLO_DIR / "labels" / split
    
    for stem in common_stems:
        img_file = img_stems[stem]
        lbl_file = lbl_stems[stem]
        
        dest_img_path = dest_img_dir / img_file.name
        dest_lbl_path = dest_lbl_dir / lbl_file.name
        
        if not dest_img_path.exists():
            shutil.copy2(img_file, dest_img_path)
        if not dest_lbl_path.exists():
            shutil.copy2(lbl_file, dest_lbl_path)
            
    audit_records.append({
        "Split": split,
        "Images Source": len(images),
        "Labels Source": len(labels),
        "Paires Valides Retenues": len(common_stems),
        "Images Écartées (Sans Label)": len(orphan_images),
        "Labels Écartés (Sans Image)": len(orphan_labels)
    })
    
    if orphan_images or orphan_labels:
        unpaired_log.append(f"=== Split {split} ===\n")
        unpaired_log.append(f"Images sans label ({len(orphan_images)}) : {[p.name for p in orphan_images[:10]]}\n")
        unpaired_log.append(f"Labels sans image ({len(orphan_labels)}) : {[p.name for p in orphan_labels[:10]]}\n\n")

# Sauvegarder le journal d'audit
with open(ARTIFACTS_DIR / "unpaired_files.txt", "w", encoding="utf-8") as f:
    f.writelines(unpaired_log if unpaired_log else ["Aucun fichier orphelin détecté : toutes les paires sont cohérentes."])

df_audit = pd.DataFrame(audit_records)
print("📊 Résumé de l'appariement et de la copie des données :\n")
display(df_audit)

## 4. Vérification de la Validité du Format YOLO et Cohérence des Classes
Exigences vérifiées pour chaque annotation `.txt` :
1. Exactement **5 valeurs** par ligne : `<class_id> <x_center> <y_center> <width> <height>`.
2. `class_id` entier valide parmi les 4 classes attendues (`0, 1, 2, 3`).
3. Coordonnées normalisées dans l'intervalle $[0.0, 1.0]$.
4. Dimensions positives (`width > 0` et `height > 0`).
5. Absence de fuite de données (*data leakage*) entre `train`, `val` et `test`.

In [ ]:
# =============================================================================
# 4. Validation Approfondie du Format YOLO
# =============================================================================
CLASS_MAPPING = {
    0: "glioma",
    1: "meningioma",
    2: "notumor",
    3: "pituitary"
}
NUM_CLASSES = len(CLASS_MAPPING)

def validate_yolo_annotations(dataset_path, num_classes=4):
    dataset_path = Path(dataset_path)
    problems = []
    stats = []
    stems_per_split = {}
    
    for split in SPLITS:
        img_dir = dataset_path / "images" / split
        lbl_dir = dataset_path / "labels" / split
        
        if not img_dir.is_dir() or not lbl_dir.is_dir():
            continue
            
        images, labels = list_split_files(img_dir, lbl_dir)
        stems_per_split[split] = {p.stem for p in images}
        
        box_count = 0
        empty_labels = 0
        class_counter = Counter()
        
        for lbl_path in labels:
            lines = [ln.strip() for ln in lbl_path.read_text(encoding="utf-8").splitlines() if ln.strip()]
            if not lines:
                empty_labels += 1
                continue
                
            for line_no, line in enumerate(lines, start=1):
                parts = line.split()
                if len(parts) != 5:
                    problems.append(f"[{split}] {lbl_path.name}:{line_no} attend 5 valeurs, reçu {len(parts)}")
                    continue
                try:
                    cid = int(parts[0])
                    x, y, w, h = map(float, parts[1:])
                except ValueError:
                    problems.append(f"[{split}] {lbl_path.name}:{line_no} valeurs non numériques")
                    continue
                    
                if cid < 0 or cid >= num_classes:
                    problems.append(f"[{split}] {lbl_path.name}:{line_no} classe ID {cid} hors bornes [0, {num_classes-1}]")
                if not all(0.0 <= v <= 1.0 for v in (x, y, w, h)):
                    problems.append(f"[{split}] {lbl_path.name}:{line_no} coordonnées non normalisées dans [0, 1]")
                if w <= 0 or h <= 0:
                    problems.append(f"[{split}] {lbl_path.name}:{line_no} largeur ou hauteur <= 0")
                    
                box_count += 1
                class_counter[cid] += 1
                
        stats.append({
            "Split": split,
            "Images": len(images),
            "Labels": len(labels),
            "Boîtes Totales": box_count,
            "Images Fond (Vides)": empty_labels,
            "Classes Détectées": sorted(class_counter.keys())
        })
        
    # Vérification du Data Leakage (chevauchement entre splits)
    split_names = list(stems_per_split.keys())
    for i, a in enumerate(split_names):
        for b in split_names[i + 1:]:
            overlap = stems_per_split[a] & stems_per_split[b]
            if overlap:
                problems.append(f"FUITE DE DONNÉES : {len(overlap)} image(s) présente(s) à la fois dans '{a}' et '{b}' !")
                
    return stats, problems

stats, validation_problems = validate_yolo_annotations(DATASET_YOLO_DIR, num_classes=NUM_CLASSES)

df_validation = pd.DataFrame(stats)
print("✅ Rapport de validation de structure et classes :\n")
display(df_validation)

if validation_problems:
    print(f"\n❌ {len(validation_problems)} anomalie(s) détectée(s) :")
    for p in validation_problems[:10]:
        print(f"  • {p}")
else:
    print("\n🎉 Intégrité 100% validée : toutes les annotations respectent rigoureusement le standard YOLOv8.")

## 5. Comptage des Images, Fichiers d'Annotation & Distribution des Classes

In [ ]:
# =============================================================================
# 5. Comptage & Distribution Statistique des Classes
# =============================================================================
box_records = []

for split in SPLITS:
    lbl_dir = DATASET_YOLO_DIR / "labels" / split
    if not lbl_dir.is_dir():
        continue
    for p in lbl_dir.glob("*.txt"):
        if p.name in IGNORED_LABEL_FILES:
            continue
        for line in p.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                cid = int(parts[0])
                box_records.append({
                    "Split": split.capitalize(),
                    "Class_ID": cid,
                    "Class_Name": CLASS_MAPPING.get(cid, f"Class_{cid}")
                })

df_boxes = pd.DataFrame(box_records)
distribution_table = pd.crosstab(df_boxes["Class_Name"], df_boxes["Split"], margins=True, margins_name="Total")
print("📋 Tableau de contingence des classes par split :\n")
display(distribution_table)

# Visualisation graphique de la répartition
plt.figure(figsize=(11, 4.5))
order = [CLASS_MAPPING[i] for i in range(NUM_CLASSES)]
sns.countplot(data=df_boxes, x="Class_Name", hue="Split", order=order, palette="crest")
plt.title("Distribution des Objets Annotés par Classe et Ensemble (Train / Val / Test)", fontsize=13, fontweight="bold")
plt.xlabel("Classe de Tumeur")
plt.ylabel("Nombre de Boîtes Délimitantes")
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "dataset_class_distribution.png", dpi=200)
plt.show()

## 6. Affichage d'un Échantillon d'Images avec Bounding Boxes et Classes
Cette fonction convertit les coordonnées YOLO normalisées $(x_c, y_c, w, h)$ en coordonnées pixels $(x_1, y_1, x_2, y_2)$ et superpose les boîtes englobantes colorées avec l'étiquette de classe correspondante.

In [ ]:
# =============================================================================
# 6. Visualisation des Boîtes Englobantes sur un Échantillon d'Images
# =============================================================================
# Palette de couleurs distinctes par classe (RGB)
CLASS_COLORS = {
    0: (230, 57, 70),    # Rouge Corail (Glioma)
    1: (69, 123, 157),   # Bleu Acier (Meningioma)
    2: (42, 157, 143),   # Vert Turquoise (No Tumor)
    3: (233, 196, 106)   # Jaune Ocre (Pituitary)
}

def plot_annotated_samples(dataset_path, split="train", num_samples=8, seed=42):
    img_dir = Path(dataset_path) / "images" / split
    lbl_dir = Path(dataset_path) / "labels" / split
    
    images, _ = list_split_files(img_dir, lbl_dir)
    # Ne garder que les images avec au moins une boîte pour la visualisation
    labeled_images = []
    for img in images:
        lbl = lbl_dir / f"{img.stem}.txt"
        if lbl.is_file() and lbl.stat().st_size > 0:
            labeled_images.append(img)
            
    rng = random.Random(seed)
    selected = rng.sample(labeled_images, min(num_samples, len(labeled_images)))
    
    cols = 4
    rows = (len(selected) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = np.array(axes).reshape(-1)
    
    for idx, (ax, img_path) in enumerate(zip(axes, selected)):
        img = cv2.imread(str(img_path))
        if img is None:
            ax.axis("off")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h_img, w_img = img.shape[:2]
        
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        for line in lbl_path.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cid, xc, yc, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            
            # Dénormalisation des coordonnées pixels
            x1 = max(0, int((xc - bw / 2) * w_img))
            y1 = max(0, int((yc - bh / 2) * h_img))
            x2 = min(w_img - 1, int((xc + bw / 2) * w_img))
            y2 = min(h_img - 1, int((yc + bh / 2) * h_img))
            
            color = CLASS_COLORS.get(cid, (255, 255, 255))
            cname = CLASS_MAPPING.get(cid, str(cid))
            
            # Rectangle de détection
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            
            # Bandeau textuel pour lisibilité optimale
            label_text = f"{cname}"
            (tw, th), baseline = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
            cv2.rectangle(img, (x1, max(0, y1 - th - 8)), (x1 + tw + 6, y1), color, -1)
            cv2.putText(img, label_text, (x1 + 3, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
            
        ax.imshow(img)
        ax.set_title(f"{img_path.name}", fontsize=10, fontweight="bold")
        ax.axis("off")
        
    for ax in axes[len(selected):]:
        ax.axis("off")
        
    plt.suptitle(f"Échantillon d'Images avec Boîtes Englobantes et Classes — {split.capitalize()}", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / f"{split}_sample_bboxes.png", dpi=200)
    plt.show()

plot_annotated_samples(DATASET_YOLO_DIR, split="train", num_samples=8, seed=42)

## 7. Création Dynamique du Fichier `data.yaml`
Le fichier `data.yaml` spécifie les chemins absolus vers les sous-dossiers `train`, `val`, et `test`, le nombre de classes (`nc: 4`) ainsi que les noms de chaque classe.

In [ ]:
# =============================================================================
# 7. Génération de data.yaml
# =============================================================================
DATA_YAML_PATH = WORK_DIR / "data.yaml"

yaml_content = {
    "path": str(DATASET_YOLO_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": NUM_CLASSES,
    "names": CLASS_MAPPING
}

with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(yaml_content, f, sort_keys=False)

print(f"📄 Fichier data.yaml généré avec succès à : {DATA_YAML_PATH}\n")
print("---" * 15)
print(open(DATA_YAML_PATH, "r").read())
print("---" * 15)

## 8. Paramètres d'Augmentation des Données pour YOLOv8
Pour les images médicales d'IRM, les augmentations doivent préserver la fidélité anatomique :
- **Jitter Colorimétrique modéré** (`hsv_h`, `hsv_s`, `hsv_v`) : simule les légères variations d'intensité et de contraste des scanners IRM.
- **Translations & Échelle** (`translate=0.1`, `scale=0.5`) : robustesse aux différentes positions et échelles crâniennes.
- **Retournement Horizontal** (`fliplr=0.5`) : anatomiquement plausible en plan axial/coronal.
- **Retournement Vertical désactivé** (`flipud=0.0`) : l'orientation tête-bas n'est pas représentative des scans cliniques.
- **Mosaic** (`mosaic=1.0`, `close_mosaic=10`) : combine 4 scans pendant le début de l'entraînement, puis est désactivé sur les 10 dernières époques pour affiner la précision des délimitations.

In [ ]:
# =============================================================================
# 8. Configuration de l'Augmentation des Données
# =============================================================================
AUGMENTATION_CONFIG = {
    "hsv_h": 0.015,       # Variation de teinte (teinte très légère pour IRM)
    "hsv_s": 0.7,         # Saturation
    "hsv_v": 0.4,         # Luminosité (simule les variations de contraste T1/T2)
    "degrees": 5.0,       # Légère rotation (+/- 5 deg)
    "translate": 0.1,     # Translation modérée (+/- 10%)
    "scale": 0.5,         # Zoom et dézoom (+/- 50%)
    "shear": 1.0,         # Cisaillement minime
    "perspective": 0.0,   # Pas de distorsion de perspective
    "flipud": 0.0,        # Pas de flip vertical (non médical)
    "fliplr": 0.5,        # Flip horizontal (symétrie cérébrale gauche/droite)
    "mosaic": 1.0,        # Mélange de 4 images
    "mixup": 0.05,        # Léger mixup
    "close_mosaic": 10    # Désactiver mosaic 10 époques avant la fin
}

print("✨ Paramètres d'augmentation configurés :")
for k, v in AUGMENTATION_CONFIG.items():
    print(f"  • {k:14s}: {v}")

## 9. Choix des Hyperparamètres & Entraînement de YOLOv8
Le choix du modèle (`yolov8s.pt` ou `yolov8m.pt`), de la taille de lot (*batch size*), du taux d'apprentissage et de l'optimiseur s'adapte automatiquement aux ressources matérielles détectées.

In [ ]:
# =============================================================================
# 9. Stratégie d'Entraînement Adaptée au Matériel & Dataset
# =============================================================================
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    if vram >= 15:
        BASE_MODEL = "yolov8m.pt"
        BATCH_SIZE = 16
    elif vram >= 8:
        BASE_MODEL = "yolov8s.pt"
        BATCH_SIZE = 16
    else:
        BASE_MODEL = "yolov8n.pt"
        BATCH_SIZE = 8
    DEVICE = 0
else:
    BASE_MODEL = "yolov8n.pt"
    BATCH_SIZE = 4
    DEVICE = "cpu"

EPOCHS = 60
PATIENCE = 15
IMG_SIZE = 640

TRAINING_HYPERPARAMETERS = {
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "imgsz": IMG_SIZE,
    "patience": PATIENCE,
    "device": DEVICE,
    "workers": min(4, os.cpu_count() or 2),
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lrf": 0.01,
    "weight_decay": 0.0005,
    "warmup_epochs": 3,
    "cos_lr": True,
    "plots": True,
    **AUGMENTATION_CONFIG
}

print(f"🚀 Modèle initialisé : {BASE_MODEL}")
print(f"   • Résolution : {IMG_SIZE}x{IMG_SIZE} | Batch : {BATCH_SIZE} | Époques : {EPOCHS} (Early Stopping patience: {PATIENCE})")
print(f"   • Optimiseur : {TRAINING_HYPERPARAMETERS['optimizer']} (lr0={TRAINING_HYPERPARAMETERS['lr0']}, lrf={TRAINING_HYPERPARAMETERS['lrf']})")

# Instanciation du modèle YOLO
model = YOLO(BASE_MODEL)

# Lancement de l'entraînement
RUN_NAME = "canscan_yolov8_run"
results = model.train(
    data=str(DATA_YAML_PATH),
    project=str(ARTIFACTS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    seed=SEED,
    **TRAINING_HYPERPARAMETERS
)

TRAIN_OUTPUT_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = TRAIN_OUTPUT_DIR / "weights" / "best.pt"
print(f"\n🏆 Poids optimaux sauvegardés : {BEST_WEIGHTS}")

### Tracé des Courbes d'Apprentissage (Loss & Métriques)

In [ ]:
# Visualisation des pertes et métriques depuis results.csv
results_csv = TRAIN_OUTPUT_DIR / "results.csv"
if results_csv.is_file():
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()
    epochs = df_res["epoch"]
    
    # Graphique des pertes
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    loss_pairs = [("box_loss", "Perte de Boîte (Box Loss)"), 
                  ("cls_loss", "Perte de Classification (Cls Loss)"), 
                  ("dfl_loss", "Perte DFL (Distribution Focal Loss)")]
    
    for ax, (k, title) in zip(axes, loss_pairs):
        if f"train/{k}" in df_res.columns:
            ax.plot(epochs, df_res[f"train/{k}"], label="Train", lw=2)
        if f"val/{k}" in df_res.columns:
            ax.plot(epochs, df_res[f"val/{k}"], label="Validation", lw=2)
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel("Époque")
        ax.set_ylabel("Loss")
        ax.grid(alpha=0.3)
        ax.legend()
    plt.suptitle("Évolution des Pertes d'Entraînement et de Validation", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / "training_loss_curves.png", dpi=200)
    plt.show()
    
    # Graphique des métriques de validation
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.2))
    metric_pairs = [("metrics/precision(B)", "Précision"), 
                    ("metrics/recall(B)", "Rappel"), 
                    ("metrics/mAP50(B)", "mAP@0.50"), 
                    ("metrics/mAP50-95(B)", "mAP@0.50:0.95")]
    
    for ax, (k, title) in zip(axes, metric_pairs):
        if k in df_res.columns:
            ax.plot(epochs, df_res[k], color="#2a9d8f", lw=2.2)
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.set_xlabel("Époque")
        ax.set_ylim(0, 1)
        ax.grid(alpha=0.3)
    plt.suptitle("Évolution des Métriques de Détection (Validation)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / "validation_metrics_curves.png", dpi=200)
    plt.show()

## 10. Évaluation Complète des Performances de Détection
Extraction des métriques clés de détection d'objets sur les ensembles de **Validation** et de **Test** :
- **Précision** (Precision)
- **Rappel** (Recall)
- **mAP@0.50** (Mean Average Precision à IoU = 0.50)
- **mAP@0.50:0.95** (Moyenne IoU de 0.50 à 0.95, métrique officielle COCO)
- Décomposition détaillée globale et par classe (`glioma`, `meningioma`, `notumor`, `pituitary`).

In [ ]:
# =============================================================================
# 10. Évaluation Détaillée sur Validation et Test
# =============================================================================
eval_model = YOLO(str(BEST_WEIGHTS))

def evaluate_split(model_obj, split="val"):
    """Évalue le modèle sur un split donné et produit un tableau détaillé."""
    val_results = model_obj.val(
        data=str(DATA_YAML_PATH),
        split=split,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        project=str(ARTIFACTS_DIR),
        name=f"eval_{split}",
        plots=True,
        exist_ok=True
    )
    
    rows = [{
        "Split": split.capitalize(),
        "Classe": "GLOBAL (Toutes)",
        "Précision": val_results.box.mp,
        "Rappel": val_results.box.mr,
        "mAP@0.50": val_results.box.map50,
        "mAP@0.50:0.95": val_results.box.map
    }]
    
    for pos, cid in enumerate(val_results.box.ap_class_index):
        p, r, ap50, ap = val_results.box.class_result(pos)
        rows.append({
            "Split": split.capitalize(),
            "Classe": val_results.names[int(cid)],
            "Précision": p,
            "Rappel": r,
            "mAP@0.50": ap50,
            "mAP@0.50:0.95": ap
        })
        
    df_table = pd.DataFrame(rows)
    df_table.to_csv(ARTIFACTS_DIR / f"{split}_metrics_table.csv", index=False)
    return val_results, df_table

val_metrics, df_val = evaluate_split(eval_model, split="val")
test_metrics, df_test = evaluate_split(eval_model, split="test")

print("📊 Métriques de Détection sur l'Ensemble de VALIDATION :")
display(df_val.round(4))

print("\n🎯 Métriques de Détection sur l'Ensemble de TEST (Généralisation) :")
display(df_test.round(4))

### Affichage des Matrices de Confusion (Test)

In [ ]:
# Affichage des matrices de confusion générées automatiquement par YOLO
eval_test_dir = ARTIFACTS_DIR / "eval_test"
cm_raw = eval_test_dir / "confusion_matrix.png"
cm_norm = eval_test_dir / "confusion_matrix_normalized.png"

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
if cm_raw.is_file():
    axes[0].imshow(plt.imread(str(cm_raw)))
    axes[0].set_title("Matrice de Confusion Brute (Test)", fontsize=12, fontweight="bold")
    axes[0].axis("off")
if cm_norm.is_file():
    axes[1].imshow(plt.imread(str(cm_norm)))
    axes[1].set_title("Matrice de Confusion Normalisée (Test)", fontsize=12, fontweight="bold")
    axes[1].axis("off")
plt.tight_layout()
plt.show()

## 11. Analyse des Résultats & Diagnostic de Généralisation
Cette section évalue formellement la robustesse du modèle :
- **Écart Train/Test** : vérifie l'absence de surapprentissage critique.
- **Écart Val/Test** : confirme que l'arrêt anticipé et la sélection d'hyperparamètres ne sont pas biaisés.
- **Équilibre Précision / Rappel** : analyse des faux positifs vs faux négatifs.
- **Diagnostic par classe** : identification des tumeurs les plus complexes.

In [ ]:
# =============================================================================
# 11. Comparaison Multi-Ensembles et Diagnostic de Généralisation
# =============================================================================
# Évaluation rapide sur train pour comparaison rigoureuse
train_metrics, df_train = evaluate_split(eval_model, split="train")

summary_comparison = pd.DataFrame({
    "Train": df_train[df_train["Classe"] == "GLOBAL (Toutes)"].iloc[0][["Précision", "Rappel", "mAP@0.50", "mAP@0.50:0.95"]],
    "Validation": df_val[df_val["Classe"] == "GLOBAL (Toutes)"].iloc[0][["Précision", "Rappel", "mAP@0.50", "mAP@0.50:0.95"]],
    "Test": df_test[df_test["Classe"] == "GLOBAL (Toutes)"].iloc[0][["Précision", "Rappel", "mAP@0.50", "mAP@0.50:0.95"]]
}).T.astype(float)

print("📊 Synthèse comparative Train vs Val vs Test :\n")
display(summary_comparison.round(4))

# Diagramme en barres groupées
plt.figure(figsize=(10, 5))
ax = summary_comparison.T.plot(kind="bar", figsize=(10, 5), colormap="viridis", edgecolor="black", alpha=0.85)
plt.title("Comparaison des Performances Globales : Train vs Validation vs Test", fontsize=13, fontweight="bold")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.grid(axis="y", linestyle="--", alpha=0.4)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "generalization_comparison_chart.png", dpi=200)
plt.show()

# Diagnostic textuel automatisé
test_row = summary_comparison.loc["Test"]
train_row = summary_comparison.loc["Train"]
val_row = summary_comparison.loc["Validation"]

findings = []
train_test_gap = train_row["mAP@0.50:0.95"] - test_row["mAP@0.50:0.95"]
val_test_gap = abs(val_row["mAP@0.50:0.95"] - test_row["mAP@0.50:0.95"])

if train_test_gap > 0.15:
    findings.append(f"⚠️ Surapprentissage détecté : écart Train - Test de {train_test_gap:.3f} sur le mAP@0.5:0.95.")
else:
    findings.append(f"✅ Excellente généralisation : écart Train - Test modéré ({train_test_gap:.3f} < 0.15).")
    
if val_test_gap <= 0.05:
    findings.append(f"✅ Forte cohérence Val / Test ({val_test_gap:.3f} <= 0.05) : le critère d'arrêt anticipé est fiable.")
else:
    findings.append(f"ℹ️ Écart Val / Test de {val_test_gap:.3f} : légères disparités distributionnelles entre val et test.")

if abs(test_row["Précision"] - test_row["Rappel"]) > 0.12:
    if test_row["Précision"] > test_row["Rappel"]:
        findings.append("ℹ️ Précision supérieure au Rappel : le modèle privilégie les détections certaines (peu de faux positifs, plus de faux négatifs).")
    else:
        findings.append("ℹ️ Rappel supérieur à la Précision : le modèle privilégie la sensibilité clinique (détecte presque toutes les lésions, quelques faux positifs).")
else:
    findings.append("✅ Parfait équilibre entre Précision et Rappel sur l'ensemble de test.")

print("\n🔍 Observations et Diagnostic de Généralisation :")
for f in findings:
    print(f"  • {f}")

## 12. Démonstration des Prédictions en Inférence (Vérité Terrain vs Prédiction)
Affichage côte-à-côte sur des images de test : la boîte englobante réelle (*Ground Truth*) et la boîte prédite par YOLOv8 avec son score de confiance.

In [ ]:
# =============================================================================
# 12. Démonstration Visuelle en Inférence
# =============================================================================
test_img_dir = DATASET_YOLO_DIR / "images" / "test"
test_lbl_dir = DATASET_YOLO_DIR / "labels" / "test"
test_images, _ = list_split_files(test_img_dir, test_lbl_dir)

rng = random.Random(SEED)
sample_test_images = rng.sample(test_images, min(4, len(test_images)))

# Prédiction avec le modèle entraîné
preds = eval_model.predict(
    source=[str(p) for p in sample_test_images],
    conf=0.25,
    iou=0.7,
    imgsz=IMG_SIZE,
    device=DEVICE,
    verbose=False
)

fig, axes = plt.subplots(len(sample_test_images), 2, figsize=(12, 5 * len(sample_test_images)))

for idx, (img_path, pred_res) in enumerate(zip(sample_test_images, preds)):
    # 1. Image avec Vérité Terrain
    gt_img = cv2.imread(str(img_path))
    gt_img = cv2.cvtColor(gt_img, cv2.COLOR_BGR2RGB)
    gh, gw = gt_img.shape[:2]
    
    lbl_file = test_lbl_dir / f"{img_path.stem}.txt"
    if lbl_file.is_file():
        for line in lbl_file.read_text(encoding="utf-8").splitlines():
            parts = line.strip().split()
            if len(parts) == 5:
                cid, xc, yc, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = max(0, int((xc - bw / 2) * gw))
                y1 = max(0, int((yc - bh / 2) * gh))
                x2 = min(gw - 1, int((xc + bw / 2) * gw))
                y2 = min(gh - 1, int((yc + bh / 2) * gh))
                cv2.rectangle(gt_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(gt_img, f"GT: {CLASS_MAPPING.get(cid, cid)}", (x1, max(20, y1 - 8)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                
    # 2. Image avec Prédiction YOLO
    pred_plot = pred_res.plot(conf=True, line_width=2)
    pred_plot = cv2.cvtColor(pred_plot, cv2.COLOR_BGR2RGB)
    
    axes[idx, 0].imshow(gt_img)
    axes[idx, 0].set_title(f"Vérité Terrain (GT) — {img_path.name}", fontsize=11, fontweight="bold")
    axes[idx, 0].axis("off")
    
    axes[idx, 1].imshow(pred_plot)
    axes[idx, 1].set_title(f"Prédiction YOLOv8 — {img_path.name}", fontsize=11, fontweight="bold")
    axes[idx, 1].axis("off")

plt.suptitle("Comparaison Qualificative : Vérité Terrain vs Prédiction YOLOv8 (Test)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / "inference_gt_vs_pred.png", dpi=200)
plt.show()

## 13. Sauvegarde des Poids, Export ONNX & Archivage des Artefacts
Regroupement de tous les artefacts nécessaires au déploiement :
- Poids `best.pt`
- Export au format standard **ONNX** (`best.onnx`) pour intégration API / production
- Fichier de configuration `data.yaml`
- Métadonnées `model_info.json` (classes, seuils recommandés, métriques de performance, date)
- Archivage ZIP complet pour téléchargement en 1 clic sur Kaggle.

In [ ]:
# =============================================================================
# 13. Exportation et Archivage Final
# =============================================================================
EXPORT_DIR = ARTIFACTS_DIR / "model_deployment"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Copie de best.pt et data.yaml
final_pt = EXPORT_DIR / "best.pt"
shutil.copy2(BEST_WEIGHTS, final_pt)
shutil.copy2(DATA_YAML_PATH, EXPORT_DIR / "data.yaml")

# 2. Export ONNX
try:
    print("⚙️ Exportation du modèle au format ONNX...")
    exported_onnx = YOLO(str(final_pt)).export(format="onnx", imgsz=IMG_SIZE, dynamic=False)
    shutil.copy2(exported_onnx, EXPORT_DIR / "best.onnx")
    print(f"✅ Modèle ONNX exporté avec succès : {EXPORT_DIR / 'best.onnx'}")
except Exception as e:
    print(f"⚠️ Export ONNX impossible : {e}")

# 3. Fiche technique model_info.json
model_metadata = {
    "model_name": "CanScan YOLOv8 Brain Tumor Detector",
    "architecture": BASE_MODEL,
    "num_classes": NUM_CLASSES,
    "classes": CLASS_MAPPING,
    "input_resolution": [IMG_SIZE, IMG_SIZE],
    "recommended_conf_threshold": 0.25,
    "recommended_iou_threshold": 0.70,
    "ultralytics_version": ultralytics.__version__,
    "created_at": datetime.now().isoformat(),
    "test_metrics": summary_comparison.loc["Test"].to_dict()
}

with open(EXPORT_DIR / "model_info.json", "w", encoding="utf-8") as f:
    json.dump(model_metadata, f, indent=2)

# 4. Création de l'archive ZIP téléchargeable
ZIP_OUTPUT_PATH = WORK_DIR / "canscan_yolov8_artifacts.zip"
print(f"\n📦 Création de l'archive ZIP téléchargeable : {ZIP_OUTPUT_PATH}...")

with zipfile.ZipFile(ZIP_OUTPUT_PATH, "w", zipfile.ZIP_DEFLATED) as zip_out:
    for root, _, files in os.walk(ARTIFACTS_DIR):
        for file in files:
            f_path = Path(root) / file
            rel_path = f_path.relative_to(ARTIFACTS_DIR)
            zip_out.write(f_path, arcname=f"artifacts/{rel_path}")

print(f"🎉 Notebook exécuté avec succès !")
print(f"📁 Artefacts complets sauvegardés dans : {ARTIFACTS_DIR}")
print(f"📥 Archive téléchargeable générée : {ZIP_OUTPUT_PATH} ({ZIP_OUTPUT_PATH.stat().st_size / (1024*1024):.2f} MB)")